In [ ]:
import json
import logging

from openeo.rest.udp import build_process_dict
from utils import udp_params, urls, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
forest_baseline_mask = udp_params.FOREST_BASELINE_DATACUBE
decimal_year_of_deforestation = udp_params.DECIMAL_YEAR_OF_DEFORESTATION_DATACUBE

spatial_extent = udp_params.SPATIAL_EXTENT

resample_spatial_resolution = udp_params.SPATIAL_RESOLUTION

cropland_probability_threshold = udp_params.CROPLAND_PROBABILITY_THRESHOLD

In [ ]:
parameters = [
    forest_baseline_mask,
    decimal_year_of_deforestation,
    spatial_extent,
    resample_spatial_resolution,
    cropland_probability_threshold,
]

# UDP

# Load Uganda ADM-4 boundaries

geoboundaries URL https://www.geoboundaries.org/api/current/gbHumanitarian/UGA/ADM4/

In [ ]:
ADM_BOUNDARIES_URL = "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/releaseData/gbHumanitarian/UGA/ADM4/geoBoundaries-UGA-ADM4.geojson"

adm_boundaries = connection.load_url(
    ADM_BOUNDARIES_URL,
    format="GeoJSON",
)

In [ ]:
adm_boundaries.metadata.dimension_names()

In [ ]:
# adm_boundaries = adm_boundaries.filter_bbox(extent=spatial_extent)
# OpenEoApiError: [400] ProcessParameterInvalid: The value passed for parameter 'data' in process 'filter_bbox' is invalid: Expected raster cube but got vector cube.

In [ ]:
vectorcube_filter_bbox_udf = openeo.UDF.from_file(
    "../udf/vectorcube_filter_bbox.py",
    runtime="Python",
    version="3.11",
    # context set here is ignored! 😠
    context={
        "spatial_extent": spatial_extent,
    },
)

In [ ]:
adm_boundaries = adm_boundaries.apply_dimension(
    process=vectorcube_filter_bbox_udf,
    dimension="geometry",
    # try passing context here as well 🤷‍♂️
    context={
        "spatial_extent": spatial_extent,
    },
)

# Load decimal year of deforestation

In [ ]:
# initial spatial filter
# also creates client-side DataCube instances

deforestation_year = connection.datacube_from_process(
    "filter_bbox",
    data=decimal_year_of_deforestation,
    extent=spatial_extent,
)

# load forest baseline

In [ ]:
# initial spatial filter
# also creates client-side DataCube instances

forest_baseline_mask = connection.datacube_from_process(
    "filter_bbox",
    data=forest_baseline_mask,
    extent=spatial_extent,
)

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
forest_baseline_mask = forest_baseline_mask.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

# load cropland probability

In [ ]:
cropland_probability = connection.load_stac(
    url=urls.MEAN_CROPS_STAC,
    spatial_extent=spatial_extent,
    bands=["crops"],
)

In [ ]:
cropland_probability = utils.drop_hidden_dimension(cropland_probability, "t")

In [ ]:
# resample onto S1 grid @ resample_spatial_resolution
cropland_probability = cropland_probability.resample_cube_spatial(
    deforestation_year,
    method="near",  # Reference implementation uses rioxarray reproject_match() which has default Resampling.nearest
)

In [ ]:
# cropland_mask = cropland_probability.band("crops") > cropland_probability_threshold

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

gt_cropland_probability_threshold = openeo.UDF.from_file(
    "../udf/binary_operator.py",
    runtime="Python",
    version="3.11",
    context={
        "operator": "gt",
        "argument": cropland_probability_threshold,
    },
)

cropland_mask = (
    cropland_probability.filter_bands("crops")
    .apply(gt_cropland_probability_threshold)
    .convert_data_type("bool")
    .band("crops")
)

# Structure of following code

The CDSE openEO backend has basically no support for processing vector data.
Ideally I would generate each KPI, then merge them together into a single output.
However, this is not possible.

https://forum.dataspace.copernicus.eu/t/merge-vector-cubes/5425/

Essentially, `aggregate_spatial` has to be the last process,
because after that the data is a vector cube, and there is no facility to process it any futher. 😡

# KPI 0: Forest stock baseline

In [ ]:
# re-label bands whilst still a raster datacube
forest_stock_baseline = forest_baseline_mask.rename_labels(
    dimension="bands",
    target=["forest_stock_baseline_ha"],
)

# KPI 1: Forest loss in each year

It would be nice if this wasn't written out for each year manually.
But it does have to be a UDP.

In [ ]:
deforestation_yr_band = deforestation_year.band("data")

In [ ]:
deforestation_2020 = (deforestation_yr_band >= 2020) & (deforestation_yr_band < 2021)
deforestation_2020_datacube = deforestation_2020.add_dimension(
    name="bands", label="deforestation_2020_ha", type="bands"
)

In [ ]:
deforestation_2021 = (deforestation_yr_band >= 2021) & (deforestation_yr_band < 2022)
deforestation_2021_datacube = deforestation_2021.add_dimension(
    name="bands", label="deforestation_2021_ha", type="bands"
)

In [ ]:
deforestation_2022 = (deforestation_yr_band >= 2022) & (deforestation_yr_band < 2023)
deforestation_2022_datacube = deforestation_2022.add_dimension(
    name="bands", label="deforestation_2022_ha", type="bands"
)


In [ ]:
deforestation_2023 = (deforestation_yr_band >= 2023) & (deforestation_yr_band < 2024)
deforestation_2023_datacube = deforestation_2023.add_dimension(
    name="bands", label="deforestation_2023_ha", type="bands"
)


In [ ]:
deforestation_2024 = (deforestation_yr_band >= 2024) & (deforestation_yr_band < 2025)
deforestation_2024_datacube = deforestation_2024.add_dimension(
    name="bands", label="deforestation_2024_ha", type="bands"
)


# KPI 2: Forest loss to cropland

In [ ]:
# in order to do band math, we need to stack all of the inputs into a single DataCube
# https://github.com/Open-EO/openeo-python-client/issues/748

band0 = cropland_mask.add_dimension("bands", label="cropland_mask", type="bands")
band1 = deforestation_2020.add_dimension(
    "bands", label="deforestation_2020", type="bands"
)
band2 = deforestation_2021.add_dimension(
    "bands", label="deforestation_2021", type="bands"
)
band3 = deforestation_2022.add_dimension(
    "bands", label="deforestation_2022", type="bands"
)
band4 = deforestation_2023.add_dimension(
    "bands", label="deforestation_2023", type="bands"
)
band5 = deforestation_2024.add_dimension(
    "bands", label="deforestation_2024", type="bands"
)

combined = (
    band0.merge_cubes(band1)
    .merge_cubes(band2)
    .merge_cubes(band3)
    .merge_cubes(band4)
    .merge_cubes(band5)
)

In [ ]:
forest_loss_to_cropland_2020 = combined.band("cropland_mask") & combined.band(
    "deforestation_2020"
)
forest_loss_to_cropland_2020_datacube = forest_loss_to_cropland_2020.add_dimension(
    name="bands", label="forest_loss_to_cropland_2020_ha", type="bands"
)

In [ ]:
forest_loss_to_cropland_2021 = combined.band("cropland_mask") & combined.band(
    "deforestation_2021"
)
forest_loss_to_cropland_2021_datacube = forest_loss_to_cropland_2021.add_dimension(
    name="bands", label="forest_loss_to_cropland_2021_ha", type="bands"
)

In [ ]:
forest_loss_to_cropland_2022 = combined.band("cropland_mask") & combined.band(
    "deforestation_2022"
)
forest_loss_to_cropland_2022_datacube = forest_loss_to_cropland_2022.add_dimension(
    name="bands", label="forest_loss_to_cropland_2022_ha", type="bands"
)

In [ ]:
forest_loss_to_cropland_2023 = combined.band("cropland_mask") & combined.band(
    "deforestation_2023"
)
forest_loss_to_cropland_2023_datacube = forest_loss_to_cropland_2023.add_dimension(
    name="bands", label="forest_loss_to_cropland_2023_ha", type="bands"
)

In [ ]:
forest_loss_to_cropland_2024 = combined.band("cropland_mask") & combined.band(
    "deforestation_2024"
)
forest_loss_to_cropland_2024_datacube = forest_loss_to_cropland_2024.add_dimension(
    name="bands", label="forest_loss_to_cropland_2024_ha", type="bands"
)

# Stack all KPI layers

In [ ]:
kpis_raster = (
    # KPI 0
    forest_stock_baseline
    # KPI 1
    .merge_cubes(deforestation_2020_datacube)
    .merge_cubes(deforestation_2021_datacube)
    .merge_cubes(deforestation_2022_datacube)
    .merge_cubes(deforestation_2023_datacube)
    .merge_cubes(deforestation_2024_datacube)
    # KPI 2
    .merge_cubes(forest_loss_to_cropland_2020_datacube)
    .merge_cubes(forest_loss_to_cropland_2021_datacube)
    .merge_cubes(forest_loss_to_cropland_2022_datacube)
    .merge_cubes(forest_loss_to_cropland_2023_datacube)
    .merge_cubes(forest_loss_to_cropland_2024_datacube)
)

convert pixel masks to pixel area (units: ha)

In [ ]:
# 1 ha = 10000 m^2
# kpis_raster = kpis_raster.apply(lambda x: x * resample_spatial_resolution * resample_spatial_resolution / 10000)

# cannot use band math with Parameters 🙁
# https://forum.dataspace.copernicus.eu/t/udp-parameter-not-applied/5282

pixel_mask_to_area_ha = openeo.UDF.from_file(
    "../udf/pixel_area_ha.py",
    runtime="Python",
    version="3.11",
    context={
        "spatial_resolution": resample_spatial_resolution,
    },
)

kpis_raster = kpis_raster.apply(pixel_mask_to_area_ha)

raster stats

In [ ]:
kpis_vector = kpis_raster.aggregate_spatial(
    geometries=adm_boundaries, reducer=openeo.processes.sum
)

# Serialise UDP

In [ ]:
summary = "Forect loss KPIs"
description = (
    "Aggregate forest-stock and deforestation KPIs by Uganda ADM-4 administrative unit, "
    "from a forest baseline mask and a decimal-year deforestation cube. "
    "1. Spatially filter ADM-4 boundaries, the forest baseline, and the deforestation "
    "cube to the requested extent, then resample the baseline onto the deforestation grid. "
    "2. Load cropland probability, resample it onto the same grid, and threshold "
    "it into a cropland mask. "
    "3. Build annual deforestation masks (2020-2024) and the subset of those "
    "pixels that overlap cropland. "
    "4. Convert each pixel mask to area in hectares and sum over ADM-4 geometries. "
    "The returned vector cube contains forest-stock baseline, annual forest loss, "
    "and annual forest-loss-to-cropland areas (ha) per administrative unit."
)

udp_spec = build_process_dict(
    kpis_vector,
    process_id="KPIs",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A vector cube, with columns (bands) for each KPI",
        "schema": {"type": "object", "subtype": "vector-cube"},
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)